# Simple ZScore

The logic is:

High mvrv_zscore (overvalued, e.g. +3) → preference = −3 → buy less
Low mvrv_zscore (undervalued, e.g. −2) → preference = +2 → buy more
If mvrv_zscore column is missing (e.g. custom data without MVRV), falls back to np.zeros → uniform buying

#### Comparison to MVRV strategy:

| "" |SimpleZScoreStrategy|MVRVStrategy|
|---|---|---|
|Preference formula|−mvrv_zscore (one line)|Weighted blend of 3 signals (70/20/10)|
|Signals used|1 (mvrv_zscore)|3 (mvrv_zscore, mvrv_percentile, price_vs_ma)|
|Modulation|None|+ gradient, acceleration, volatility dampening|
|Purpose|Teaching example / baseline|Production strategy|

**example:**
&ensp;Aug 2, 2016 — price = $542.31, zscore = 0.1 <br>This day is somewhere in the middle of a 365-day window. Let's say it's day index d=180 (day 181 of 365).

**Step A — preference:**
preference = −0.1 <br>(zscore = 0.1 means Bitcoin is slightly overvalued -> mild signal to buy less)

**Step B — raw:**
$raw_{180} = \frac{1}{365} \exp{-0.1}$ = 0.002740×0.9048 = 0.002479 <br>For comparison, a perfectly neutral day (zscore = 0) would have:
$raw_{neutral} = \frac{1}{365} \exp{0}$ <br>= 0.002740

**Step C — stable_signal:**
Suppose the preceding 180 days had an average zscore of +0.8 (overvalued period). Then their raw values averaged around: <br>running_mean ≈ $\frac{1}{365} \exp{-0.8}$ ≈ 0.002740×0.449 = 0.001230
Then: <br>$signal_{180} = \frac{0.002479}{0.001230}$ ≈ 2.016 <br>Even though zscore = 0.1 means slightly overvalued, relative to the preceding very-overvalued days, this day looks like a buying opportunity.

**Step D — proposed:**
$proposed_{180} = 2.016 × \frac{1}{365}$ = 0.005523

**Step E — clipped:**
With 184 days remaining after this day, budget constraints give roughly: <br>max_upper = min(0.1, remaining − 184×0.00001) ≈ 0.1 <br>min_lower = max(0.00001, remaining − 184×0.1) <br>The proposed 0.005523 is within bounds → weight = 0.0055 (approximately)

In [1]:
import sys
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from stacksats.model_development import precompute_features
from stacksats.runner import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.signals.simple_zscore import SimpleZScoreStrategy
from stacksats.strategies.stable.baselines.uniform import UniformStrategy

_root = Path.cwd()
while not (_root / "src").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src import config, data_utils, plots, strategy_utils

STACKSATS_DATA_PATH = config.STACKSATS_DATA_PATH
RAW_PATH = config.RAW_PATH
total_budget_usd = config.TOTAL_BUDGET_USD

data_utils.check_stacksats_data(STACKSATS_DATA_PATH, RAW_PATH)
# Load raw BTC data
btc_df = pl.read_parquet(STACKSATS_DATA_PATH).sort("date")
btc_df = btc_df.with_columns(pl.col("date").cast(pl.Datetime))


In [2]:
btc_train = (
    btc_df
    .filter(
        (pl.col("date") >= pl.datetime(2010, 8, 16)) &
        (pl.col("date") <= pl.datetime(2023, 12, 31)) &
        pl.col("price_usd").is_not_null()
    )
    .sort("date")
)

print("Filtered rows:", btc_train.height)
print(
    btc_train.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Filtered rows: 4886
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-08-16 00:00:00 ┆ 2023-12-31 00:00:00 │
└─────────────────────┴─────────────────────┘


In [3]:
# Strategy runner
runner = StrategyRunner()

cycle_results = {}

for cycle in plots.calendar_cycles:
    result = strategy_utils.process_cycle_year_by_year(
        cycle=cycle,
        btc_data=btc_train,
        runner=runner,
        dynamic_strategy=SimpleZScoreStrategy(),
        total_budget_usd=total_budget_usd,
        top_buy_quantile=0.90
    )

    cycle_results[cycle["label"]] = result


Processing Cycle 1: 2010-2013
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2010-08-16 00:00:00 ┆ 2013-12-31 00:00:00 ┆ 1234 │
└─────────────────────┴─────────────────────┴──────┘
2010: skipped, less than 365 rows
2010: skipped, less than 365 rows
2011: exported 365 rows
2011: exported 365 rows
2012: exported 365 rows
2012: exported 365 rows
2013: exported 365 rows
2013: exported 365 rows

Processing Cycle 2: 2014-2017
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2014-01-01 00:00:00 ┆ 2017-12-31 00:00:00 ┆ 1461 │
└────────

In [4]:
cols = plots.StrategyColumns(
    weight="dynamic_weight",
    spd="sats_per_dollar_dynamic",
    sats_accum="sats_accum_dynamic",
)

# Combine all 4 cycles into one DataFrame
combined_plot_df = pd.concat(
    [r["plot_df"] for r in cycle_results.values()]
).sort_values("date").reset_index(drop=True)

# per cycle plots
cycle_plots = plots.plot_strategy_by_cycle(combined_plot_df, cols, "Simple Z-Score")
plt.show()

In [5]:
# per year plots
year_plots = plots.plot_strategy_by_year(combined_plot_df, cols, "Simple Z-Score")
plt.show()

In [6]:
zscore_strategy = SimpleZScoreStrategy()
uniform_strategy = UniformStrategy()

zscore_yearly = []
uniform_yearly = []

for year in range(2018, 2024):
    zscore_result = strategy_utils.export_one_year(zscore_strategy, btc_train, year, runner)
    if zscore_result is not None:
        zscore_yearly.append(zscore_result)

    uniform_result = strategy_utils.export_one_year(uniform_strategy, btc_train, year, runner)
    if uniform_result is not None:
        uniform_yearly.append(uniform_result)

print("Simple Z-Score valid yearly exports:", len(zscore_yearly))
print("Uniform valid yearly exports:", len(uniform_yearly))

2018: exported 365 rows
2018: exported 365 rows
2019: exported 365 rows
2019: exported 365 rows
2020: exported 365 rows
2020: exported 365 rows
2021: exported 365 rows
2021: exported 365 rows
2022: exported 365 rows
2022: exported 365 rows
2023: exported 365 rows
2023: exported 365 rows
Simple Z-Score valid yearly exports: 6
Uniform valid yearly exports: 6


In [7]:
if not zscore_yearly:
    raise ValueError("No valid Simple Z-Score exports were produced.")

if not uniform_yearly:
    raise ValueError("No valid Uniform exports were produced.")

zscore_all = pl.concat(zscore_yearly).rename({"weight": "zscore_weight_raw"})
uniform_all = pl.concat(uniform_yearly).rename({"weight": "baseline_weight_raw"})

print("Simple Z-Score combined rows:", zscore_all.height)
print("Uniform combined rows:", uniform_all.height)

Simple Z-Score combined rows: 2190
Uniform combined rows: 2190


In [8]:
merged = (
    zscore_all
    .select(["date", "price_usd", "zscore_weight_raw"])
    .join(
        uniform_all.select(["date", "baseline_weight_raw"]),
        on="date",
        how="inner"
    )
    .sort("date")
)

print("Merged rows:", merged.height)

merged.head()

Merged rows: 2190


date,price_usd,zscore_weight_raw,baseline_weight_raw
datetime[μs],f64,f64,f64
2018-01-01 00:00:00,13466.31,0.00274,0.00274
2018-01-02 00:00:00,14888.11,0.00274,0.00274
2018-01-03 00:00:00,15098.14,0.00274,0.00274
2018-01-04 00:00:00,15144.99,0.00274,0.00274
2018-01-05 00:00:00,16960.01,0.00274,0.00274


In [9]:
#zscore_sum = merged["zscore_weight_raw"].sum()
#baseline_sum = merged["baseline_weight_raw"].sum()

# merged = merged.with_columns([
#     (pl.col("zscore_weight_raw") / zscore_sum).alias("zscore_weight"),
#     (pl.col("baseline_weight_raw") / baseline_sum).alias("baseline_weight"),
# ])

merged = merged.with_columns(pl.col("date").dt.year().alias("year"))

merged = merged.with_columns([
    (pl.col("zscore_weight_raw") / pl.col("zscore_weight_raw").sum().over("year")).alias("zscore_weight"),
    (pl.col("baseline_weight_raw") / pl.col("baseline_weight_raw").sum().over("year")).alias("baseline_weight"),
])

print("Simple Z-Score normalized weight sum:", merged["zscore_weight"].sum())
print("Baseline normalized weight sum:", merged["baseline_weight"].sum())


Simple Z-Score normalized weight sum: 6.0
Baseline normalized weight sum: 6.0


In [10]:
merged = merged.with_columns([
    (pl.col("zscore_weight") * total_budget_usd).alias("dynamic_usd"),
    (pl.col("baseline_weight") * total_budget_usd).alias("baseline_usd"),
])

merged = merged.with_columns([
    (pl.col("dynamic_usd") / pl.col("price_usd")).alias("btc_accum_dynamic"),
    (pl.col("baseline_usd") / pl.col("price_usd")).alias("btc_accum_baseline"),
])

merged = merged.with_columns([
    (pl.col("btc_accum_dynamic") * 100_000_000).alias("sats_accum_dynamic"),
    (pl.col("btc_accum_baseline") * 100_000_000).alias("sats_accum_baseline"),
])

merged = merged.with_columns([
    (pl.col("sats_accum_dynamic") / pl.col("dynamic_usd")).alias("sats_per_dollar_dynamic"),
    (pl.col("sats_accum_baseline") / pl.col("baseline_usd")).alias("sats_per_dollar_baseline"),
])

total_dynamic_btc = merged["btc_accum_dynamic"].sum()
total_baseline_btc = merged["btc_accum_baseline"].sum()

num_years = merged["date"].dt.year().n_unique()
total_invested = total_budget_usd * num_years

sats_per_dollar_dynamic = (total_dynamic_btc / total_invested) * 100_000_000
sats_per_dollar_baseline = (total_baseline_btc / total_invested) * 100_000_000
# sats_per_dollar_dynamic = (total_dynamic_btc / total_budget_usd) * 100_000_000
# sats_per_dollar_baseline = (total_baseline_btc / total_budget_usd) * 100_000_000

pct_diff_vs_baseline = (
    (total_dynamic_btc - total_baseline_btc) / total_baseline_btc
) * 100

performance_label = "better" if pct_diff_vs_baseline > 0 else "worse"

print(f"Total BTC accumulated, Simple Z-Score: {total_dynamic_btc:.6f}")
print(f"Total BTC accumulated, DCA: {total_baseline_btc:.6f}")
print(f"Sats per dollar, Simple Z-Score: {sats_per_dollar_dynamic:.2f}")
print(f"Sats per dollar, DCA: {sats_per_dollar_baseline:.2f}")
print(f"Simple Z-Score performed {abs(pct_diff_vs_baseline):.2f}% {performance_label} than DCA")

Total BTC accumulated, Simple Z-Score: 0.484355
Total BTC accumulated, DCA: 0.505128
Sats per dollar, Simple Z-Score: 8072.58
Sats per dollar, DCA: 8418.79
Simple Z-Score performed 4.11% worse than DCA


In [11]:
plot_df = merged.to_pandas()
plot_df["date"] = pd.to_datetime(plot_df["date"])
plot_df["year"] = plot_df["date"].dt.year
plot_df["cycle_label"] = plot_df["date"].apply(plots.assign_cycle_label)

plot_df.head()

,date,price_usd,zscore_weight_raw,baseline_weight_raw,year,zscore_weight,baseline_weight,dynamic_usd,baseline_usd,btc_accum_dynamic,btc_accum_baseline,sats_accum_dynamic,sats_accum_baseline,sats_per_dollar_dynamic,sats_per_dollar_baseline,cycle_label
0,2018-01-01,13466.31,0.00274,0.00274,2018,0.00274,0.00274,2.739726,2.739726,0.000203,0.000203,20345.039045,20345.039045,7425.939251,7425.939251,Cycle 3: 2018-2021
1,2018-01-02,14888.11,0.00274,0.00274,2018,0.00274,0.00274,2.739726,2.739726,0.000184,0.000184,18402.107638,18402.107638,6716.769288,6716.769288,Cycle 3: 2018-2021
2,2018-01-03,15098.14,0.00274,0.00274,2018,0.00274,0.00274,2.739726,2.739726,0.000181,0.000181,18146.116193,18146.116193,6623.332410,6623.332410,Cycle 3: 2018-2021
3,2018-01-04,15144.99,0.00274,0.00274,2018,0.00274,0.00274,2.739726,2.739726,0.000181,0.000181,18089.982413,18089.982413,6602.843581,6602.843581,Cycle 3: 2018-2021
4,2018-01-05,16960.01,0.00274,0.00274,2018,0.00274,0.00274,2.739726,2.739726,0.000162,0.000162,16154.035448,16154.035448,5896.222939,5896.222939,Cycle 3: 2018-2021


## Adding Test data

In [12]:
# Testing window: 2024 onwards (held-out period)
btc_test = (
    btc_df
    .filter(
        (pl.col("date") >= pl.datetime(2024, 1, 1)) &
        pl.col("price_usd").is_not_null()
    )
    .sort("date")
)

print("Test rows:", btc_test.height)
print(
    btc_test.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Test rows: 803
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2024-01-01 00:00:00 ┆ 2026-03-13 00:00:00 │
└─────────────────────┴─────────────────────┘


In [13]:
zscore_test_yearly = []
uniform_test_yearly = []

test_years = sorted(btc_test["date"].dt.year().unique().to_list())
print("Test years available:", test_years)

for year in test_years:
    zscore_result = strategy_utils.export_one_year(zscore_strategy, btc_test, year, runner)
    if zscore_result is not None:
        zscore_test_yearly.append(zscore_result)

    uniform_result = strategy_utils.export_one_year(uniform_strategy, btc_test, year, runner)
    if uniform_result is not None:
        uniform_test_yearly.append(uniform_result)

print("Simple Z-Score test exports:", len(zscore_test_yearly))
print("Uniform test exports:", len(uniform_test_yearly))

Test years available: [2024, 2025, 2026]
2024: exported 365 rows
2024: exported 365 rows
2025: exported 365 rows
2025: exported 365 rows
2026: skipped, less than 365 rows
2026: skipped, less than 365 rows
Simple Z-Score test exports: 2
Uniform test exports: 2


In [14]:
# Build test merged dataframe (same pipeline as train)
zscore_test_all = pl.concat(zscore_test_yearly).rename({"weight": "zscore_weight_raw"})
uniform_test_all = pl.concat(uniform_test_yearly).rename({"weight": "baseline_weight_raw"})

merged_test = (
    zscore_test_all
    .select(["date", "price_usd", "zscore_weight_raw"])
    .join(uniform_test_all.select(["date", "baseline_weight_raw"]), on="date", how="inner")
    .sort("date")
)

merged_test = merged_test.with_columns(pl.col("date").dt.year().alias("year"))
merged_test = merged_test.with_columns([
    (pl.col("zscore_weight_raw") / pl.col("zscore_weight_raw").sum().over("year")).alias("zscore_weight"),
    (pl.col("baseline_weight_raw") / pl.col("baseline_weight_raw").sum().over("year")).alias("baseline_weight"),
])
merged_test = merged_test.with_columns([
    (pl.col("zscore_weight") * total_budget_usd).alias("dynamic_usd"),
    (pl.col("baseline_weight") * total_budget_usd).alias("baseline_usd"),
])
merged_test = merged_test.with_columns([
    (pl.col("dynamic_usd") / pl.col("price_usd")).alias("btc_accum_dynamic"),
    (pl.col("baseline_usd") / pl.col("price_usd")).alias("btc_accum_baseline"),
])
merged_test = merged_test.with_columns([
    (pl.col("btc_accum_dynamic") * 1e8).alias("sats_accum_dynamic"),
    (pl.col("btc_accum_baseline") * 1e8).alias("sats_accum_baseline"),
])
merged_test = merged_test.with_columns([
    (pl.col("sats_accum_dynamic") / pl.col("dynamic_usd")).alias("sats_per_dollar_dynamic"),
    (pl.col("sats_accum_baseline") / pl.col("baseline_usd")).alias("sats_per_dollar_baseline"),
])

test_plot_df = merged_test.to_pandas()
test_plot_df["date"] = pd.to_datetime(test_plot_df["date"])
test_plot_df["year"] = test_plot_df["date"].dt.year
test_plot_df["cycle_label"] = test_plot_df["date"].apply(plots.assign_cycle_label)

In [15]:
combined_plot_df = pd.concat([plot_df, test_plot_df]).sort_values("date").reset_index(drop=True)

cols = plots.StrategyColumns(
    weight="zscore_weight",
    spd="sats_per_dollar_dynamic",
    sats_accum="sats_accum_dynamic",
)

# Full period
full_plot = plots.plot_strategy_full_period(
    combined_plot_df, 
    cols, 
    "Simple Z-Score", 
    date_range=("2018-01-01", str(combined_plot_df["date"].max().date())),
    test_start_date="2024-01-01"
)
plt.show()

## Full Window: 2010–2023 - Not to be used

A single chart spanning the entire date range with Bitcoin halving cycle bands shaded in the background.  Heavy buy days (top 10% of SimpleZScore allocation weights globally) are marked as red circles.

In [16]:
# Precompute MVRV z-score features for hover tooltips
features_df = precompute_features(btc_df).select(["date", "mvrv_zscore"])

def _dedup(batch, weight_col: str) -> pl.DataFrame:
    """Flatten overlapping rolling windows to one row per date.

    When a strategy runs over a multi-year range the runner produces
    overlapping rolling 365-day windows internally.  Each window assigns
    its own weight to every date it covers, so a single date may appear in
    dozens of windows with slightly different weights.  _dedup keeps the
    first weight seen for each date.
    """
    return (
        batch.to_dataframe()
        .group_by("date")
        .agg(
            pl.first("weight").alias(weight_col),
            pl.first("price_usd").alias("price_usd"),
        )
        .sort("date")
    )

In [17]:
WINDOW_START = "2010-01-01"
WINDOW_END   = "2023-12-31"
config = ExportConfig(range_start=WINDOW_START, range_end=WINDOW_END)

zscore_dd  = _dedup(runner.export(SimpleZScoreStrategy(), config, btc_df=btc_df), "zscore_weight")
uniform_dd = (
    _dedup(runner.export(UniformStrategy(), config, btc_df=btc_df), "baseline_weight")
    .select(["date", "baseline_weight"])
)

merged_full = zscore_dd.join(uniform_dd, on="date", how="inner")

zscore_w_sum    = merged_full["zscore_weight"].sum()
baseline_w_sum  = merged_full["baseline_weight"].sum()

merged_full = merged_full.with_columns([
    (1e8 / pl.col("price_usd")).alias("sats_per_dollar_dynamic"),
    (1e8 / pl.col("price_usd")).alias("sats_per_dollar_baseline"),
    ((pl.col("zscore_weight")   / zscore_w_sum   * total_budget_usd) / pl.col("price_usd") * 1e8).alias("sats_accum_dynamic"),
    ((pl.col("baseline_weight") / baseline_w_sum * total_budget_usd) / pl.col("price_usd") * 1e8).alias("sats_accum_baseline"),
])


zscore_spd  = (merged_full["zscore_weight"]  * merged_full["sats_per_dollar_dynamic"]).sum() / merged_full["zscore_weight"].sum()
baseline_spd = (merged_full["baseline_weight"] * merged_full["sats_per_dollar_baseline"]).sum() / merged_full["baseline_weight"].sum()
excess_pct  = (zscore_spd - baseline_spd) / baseline_spd * 100

threshold = merged_full["zscore_weight"].quantile(0.90)
merged_full = merged_full.join(features_df, on="date", how="left").with_columns(
    (pl.col("zscore_weight") >= threshold).alias("heavy_buy")
)
df = merged_full.to_pandas()

cols = plots.StrategyColumns(
    weight="zscore_weight",
    spd="sats_per_dollar_dynamic",
    sats_accum="sats_accum_dynamic",
)

# Full period
full_plot = plots.plot_strategy_full_period(df, cols, "Simple Z-Score", ("2010-08-16", "2023-12-31"))
plt.show()
